# Recognizing speech - Portfolio assignment 1 (Digits)

**Dataset:** `x_digits.npy` and `y_digits.npy`.

**Goal:** classify spoken digits `0..9` from log-spectrograms (`129 x 71`).

**Implemented:**
- Model A: Fully Connected Network (MLP)
- Model B: Convolutional Neural Network (CNN)
- Hyperparameter tuning (manual small grid)
- Train/validation/test accuracy + learning curves + confusion matrices


## Required outputs to include
- For BOTH models: accuracy on **train**, **validation**, and **test** sets.
- Short description of the approach and hyperparameter tuning.
- At least two interesting figures: learning curves and confusion matrices are included.


In [ ]:
%matplotlib inline

import os
import random
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


: 

In [ ]:
# Paths 
X_PATH = "x_digits.npy"
Y_PATH = "y_digits.npy"

if not os.path.exists(X_PATH) or not os.path.exists(Y_PATH):
    raise FileNotFoundError("x_digits.npy and/or y_digits.npy not found in the current folder.")

X_raw = np.load(X_PATH).astype(np.float32)  # (N, 129, 71)
y = np.load(Y_PATH).astype(np.int64)       # (N,)

print("X shape:", X_raw.shape)
print("y shape:", y.shape)
print("classes:", np.unique(y).tolist())
print("X min/max:", float(X_raw.min()), float(X_raw.max()))

# Class distribution plot
counts = np.bincount(y, minlength=10)
plt.figure(figsize=(7, 4))
plt.bar(np.arange(10), counts)
plt.xticks(np.arange(10))
plt.xlabel("digit class")
plt.ylabel("count")
plt.title("Class distribution")
plt.show()


In [ ]:
# 70% train, 15% val, 15% test 
N = len(X_raw)
idx_all = np.arange(N)

idx_train, idx_temp, y_train, y_temp = train_test_split(
    idx_all, y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

X_train_raw = X_raw[idx_train]
X_val_raw = X_raw[idx_val]
X_test_raw = X_raw[idx_test]

print("Train:", X_train_raw.shape, y_train.shape)
print("Val:  ", X_val_raw.shape, y_val.shape)
print("Test: ", X_test_raw.shape, y_test.shape)

# Normalize using train statistics only (per-pixel mean/std)
mean = X_train_raw.mean(axis=0, keepdims=True)
std = X_train_raw.std(axis=0, keepdims=True) + 1e-8

X_train = (X_train_raw - mean) / std
X_val = (X_val_raw - mean) / std
X_test = (X_test_raw - mean) / std

print("Normalized ranges (train):", float(X_train.min()), float(X_train.max()))


In [ ]:
def plot_learning_curves(history, title=""):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history["train_loss"], label="train_loss")
    plt.plot(history["val_loss"], label="val_loss")
    plt.title(title + " Loss")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history["train_acc"], label="train_acc")
    plt.plot(history["val_acc"], label="val_acc")
    plt.title(title + " Accuracy")
    plt.xlabel("epoch")
    plt.ylabel("accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()


def plot_confusion(y_true, y_pred, title=""):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(10)))
    plt.figure(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(range(10)))
    disp.plot(values_format="d", cmap="Blues", ax=plt.gca(), colorbar=False)
    plt.title(title)
    plt.show()


def plot_sample_predictions(X_show, y_true, y_pred, n=12, title=""):
    idx = np.random.choice(len(X_show), size=min(n, len(X_show)), replace=False)
    plt.figure(figsize=(12, 8))
    for i, k in enumerate(idx):
        ax = plt.subplot(3, 4, i + 1)
        ax.imshow(X_show[k], aspect="auto", origin="lower")
        ax.set_title(f"t={y_true[k]}\np={y_pred[k]}")
        ax.axis("off")
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


class SpectrogramDataset(Dataset):
    def __init__(self, X, y, add_channel=False):
        # X: (N, 129, 71)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.add_channel = add_channel

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.add_channel:
            # -> (1, 129, 71)
            x = x.unsqueeze(0)
        return x, self.y[idx]


In [ ]:
def evaluate_classification(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += float(loss.item()) * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += int((preds == yb).sum().item())
            total += xb.size(0)

    return total_loss / total, correct / total


def predict_labels(model, loader):
    model.eval()
    all_preds = []
    all_true = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.append(preds)
            all_true.append(yb.numpy())
    return np.concatenate(all_true), np.concatenate(all_preds)


def train_with_early_stopping(model, train_loader, val_loader, *, epochs, lr, weight_decay, patience):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    best_val_acc = -1.0
    best_state = None
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += float(loss.item()) * xb.size(0)
            preds = torch.argmax(logits, dim=1)
            correct += int((preds == yb).sum().item())
            total += xb.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        val_loss, val_acc = evaluate_classification(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"  epoch {epoch:02d} | train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        if val_acc > best_val_acc + 1e-6:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print("  early stopping")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history


In [ ]:
# Model A: MLP

class MLP(nn.Module):
    def __init__(self, hidden_units=256, dropout=0.3):
        super().__init__()
        self.flatten = nn.Flatten()
        self.net = nn.Sequential(
            nn.Linear(129 * 71, hidden_units),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_units),
            nn.Dropout(dropout),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_units),
            nn.Dropout(dropout),
            nn.Linear(hidden_units, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.net(x)


train_ds_mlp = SpectrogramDataset(X_train, y_train, add_channel=False)
val_ds_mlp = SpectrogramDataset(X_val, y_val, add_channel=False)
test_ds_mlp = SpectrogramDataset(X_test, y_test, add_channel=False)

mlp_configs = [
    {"hidden_units": 256, "dropout": 0.3, "lr": 1e-3, "batch_size": 128},
    {"hidden_units": 512, "dropout": 0.4, "lr": 5e-4, "batch_size": 128},
]

mlp_runs = []
best_mlp = None
best_mlp_val_acc = -1.0

for i, cfg in enumerate(mlp_configs, start=1):
    print(f"MLP run {i}/{len(mlp_configs)}: {cfg}")

    train_loader = DataLoader(train_ds_mlp, batch_size=cfg["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds_mlp, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds_mlp, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)

    model = MLP(hidden_units=cfg["hidden_units"], dropout=cfg["dropout"])
    model, history = train_with_early_stopping(
        model,
        train_loader,
        val_loader,
        epochs=30,
        lr=cfg["lr"],
        weight_decay=1e-4,
        patience=6,
    )

    train_loss, train_acc = evaluate_classification(model, train_loader)
    val_loss, val_acc = evaluate_classification(model, val_loader)
    test_loss, test_acc = evaluate_classification(model, test_loader)

    print(f"  FINAL -> train_acc={train_acc:.4f} val_acc={val_acc:.4f} test_acc={test_acc:.4f}")

    mlp_runs.append(
        {
            "config": cfg,
            "history": history,
            "train_acc": float(train_acc),
            "val_acc": float(val_acc),
            "test_acc": float(test_acc),
            "model": model,
        }
    )

    if val_acc > best_mlp_val_acc:
        best_mlp_val_acc = val_acc
        best_mlp = model

best_mlp_info = sorted(mlp_runs, key=lambda r: r["val_acc"], reverse=True)[0]
plot_learning_curves(best_mlp_info["history"], title="MLP (best)")

best_batch_size = 128
best_mlp_test_loader = DataLoader(test_ds_mlp, batch_size=best_batch_size, shuffle=False, num_workers=0)
y_true, y_pred = predict_labels(best_mlp, best_mlp_test_loader)
plot_confusion(y_true, y_pred, title="MLP Confusion Matrix (test)")
plot_sample_predictions(X_test_raw, y_true, y_pred, n=12, title="MLP predictions (sample) [test]")

print("MLP best run:", {k: best_mlp_info[k] for k in ["config", "train_acc", "val_acc", "test_acc"]})


In [ ]:
# Model B: CNN

class CNN(nn.Module):
    def __init__(self, filters=(16, 32, 64), dense_units=128, dropout=0.4):
        super().__init__()
        f1, f2, f3 = filters
        self.features = nn.Sequential(
            nn.Conv2d(1, f1, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(f1),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(f1, f2, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(f2),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(f2, f3, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(f3),
        )

        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(f3, dense_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dense_units, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        return self.classifier(x)


train_ds_cnn = SpectrogramDataset(X_train, y_train, add_channel=True)
val_ds_cnn = SpectrogramDataset(X_val, y_val, add_channel=True)
test_ds_cnn = SpectrogramDataset(X_test, y_test, add_channel=True)

cnn_configs = [
    {"filters": (16, 32, 64), "dense_units": 128, "dropout": 0.4, "lr": 1e-3, "batch_size": 64},
    {"filters": (32, 64, 64), "dense_units": 128, "dropout": 0.5, "lr": 5e-4, "batch_size": 64},
]

cnn_runs = []
best_cnn = None
best_cnn_val_acc = -1.0

for i, cfg in enumerate(cnn_configs, start=1):
    print(f"CNN run {i}/{len(cnn_configs)}: {cfg}")

    train_loader = DataLoader(train_ds_cnn, batch_size=cfg["batch_size"], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds_cnn, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds_cnn, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)

    model = CNN(filters=cfg["filters"], dense_units=cfg["dense_units"], dropout=cfg["dropout"])
    model, history = train_with_early_stopping(
        model,
        train_loader,
        val_loader,
        epochs=35,
        lr=cfg["lr"],
        weight_decay=1e-4,
        patience=7,
    )

    train_loss, train_acc = evaluate_classification(model, train_loader)
    val_loss, val_acc = evaluate_classification(model, val_loader)
    test_loss, test_acc = evaluate_classification(model, test_loader)

    print(f"  FINAL -> train_acc={train_acc:.4f} val_acc={val_acc:.4f} test_acc={test_acc:.4f}")

    cnn_runs.append(
        {
            "config": cfg,
            "history": history,
            "train_acc": float(train_acc),
            "val_acc": float(val_acc),
            "test_acc": float(test_acc),
            "model": model,
        }
    )

    if val_acc > best_cnn_val_acc:
        best_cnn_val_acc = val_acc
        best_cnn = model

best_cnn_info = sorted(cnn_runs, key=lambda r: r["val_acc"], reverse=True)[0]
plot_learning_curves(best_cnn_info["history"], title="CNN (best)")

best_batch_size = 64
best_cnn_test_loader = DataLoader(test_ds_cnn, batch_size=best_batch_size, shuffle=False, num_workers=0)
y_true, y_pred = predict_labels(best_cnn, best_cnn_test_loader)
plot_confusion(y_true, y_pred, title="CNN Confusion Matrix (test)")
plot_sample_predictions(X_test_raw, y_true, y_pred, n=12, title="CNN predictions (sample) [test]")

print("CNN best run:", {k: best_cnn_info[k] for k in ["config", "train_acc", "val_acc", "test_acc"]})


## Final comparison summary

Fill in using the values printed above after running the cells.

- MLP best run:
  - Train accuracy: `0.9787`
  - Validation accuracy: `0.8866`
  - Test accuracy: `0.8803`

- CNN best run:
  - Train accuracy: `0.9833`
  - Validation accuracy: `0.9594`
  - Test accuracy: `0.9557`

Notes to include:
- The CNN outperforms the MLP on validation and test accuracy because it exploits the 2D structure of spectrograms (local time-frequency patterns learned by convolutional filters).
- The MLP shows stronger overfitting (larger gap between train and validation accuracy: ~0.092) compared to the CNN (gap: ~0.024).
- Hyperparameter tuning: for the MLP, larger capacity (512 hidden units) with moderate dropout and a smaller learning rate improved validation accuracy; for the CNN, the selected configuration (16/32/64 filters with dropout 0.4 and lr=1e-3) gave the best generalization.
